# 0. Set up

In [1]:
import os

wd_path = "/home/bastebun/work/project_paaral"
if os.getcwd() != wd_path:
    os.chdir(wd_path)
else:
    pass

In [2]:
import re
import sys
import copy
import math
import time
import pickle
import numpy as np
import pandas as pd
import multiprocessing
from collections import Counter
import matplotlib.pyplot as plt

from config.config import Config
cf = Config()

import importlib
def reload_module(module):
    importlib.reload(module)

# 1. Load data

## 1.1. Enrollment SY 22-23 to 24-25

### Preprocess

In [3]:
# !ls 'data/processed'

In [4]:
# dirpath = cf.get_path('processed_data')
# enr_fnames = [file for file in os.listdir(dirpath) if re.search(r'school_level', file, flags=re.IGNORECASE)]
# enr_fpaths = [os.path.join(dirpath, fname) for fname in enr_fnames]
# school_yrs = [re.findall(r'(.*?)_school', fname, flags=re.IGNORECASE)[0] for fname in enr_fnames]

# dfs_csv = []
# for path, yr in zip(enr_fpaths, school_yrs):
#     df_csv = pd.read_csv(path)
#     df_csv['school_year'] = yr
#     dfs_csv.append(df_csv)

# dfs_concat = pd.concat(dfs_csv, ignore_index=True)
# print(dfs_concat.shape)

In [5]:
# # Drop columns with male and female
# dfs_a = dfs_concat.copy()

# drop_cols = [col for col in dfs_a.columns if re.search(r'male$', col, flags=re.IGNORECASE)]
# dfs_a = dfs_a.drop(columns=drop_cols)
# display(dfs_a.head())

In [6]:
# stime = time.time()

# id_vars = ['school_id','old_region','division','sector','school_year']
# value_vars = dfs_a.loc[:, 'k_total':'g12_total'].columns.tolist()
# dfs_melt = dfs_a.melt(
#     id_vars=id_vars,
#     value_vars=value_vars,
#     value_name='enrollment',
#     var_name='grade_level'
# )

# uniq_lvls = dfs_melt['grade_level'].unique().tolist()
# lvls_only = [re.findall(r'(.*?)_total', label)[0] for label in uniq_lvls]
# replace_dict = {k: v for k,v in zip(uniq_lvls, lvls_only)}

# dfs_melt['grade_level'] = dfs_melt['grade_level'].replace(replace_dict)
# dfs_melt['grade_level'] = dfs_melt['grade_level'].replace(to_replace=r'g', value='Grade ', regex=True)
# dfs_melt['grade_level'] = dfs_melt['grade_level'].replace('k', 'Kindergarten')

# dfs_melt['school_year'] = dfs_melt['school_year'].replace(
#     {
#         'SY_2022_2023':'SY 2022-2023',
#         'SY_2023_2024':'SY 2023-2024',
#         'SY_2024_2025':'SY 2024-2025',
#     }
# )
# dfs_melt['school_id'] = dfs_melt['school_id'].astype('string')

# etime = np.round(time.time() - stime, decimals=2)
# print(f"Time elapsed: {etime} seconds")

# print(dfs_melt.shape)

In [7]:
# display(dfs_melt.head(3))

### Save parquet

In [8]:
# %%time
# dfs_melt.to_parquet(
#     'output/enrollment_schoolLevel_3yr.parquet',
#     index='school_year'
# )

In [9]:
# !ls -lhsa 'output'

### Load parquet

In [3]:
%%time
df_enr = pd.read_parquet('output/enrollment_schoolLevel_3yr.parquet')
print(df_enr.shape)
display(df_enr.head(3))

(2344368, 7)


,school_id,old_region,division,sector,school_year,grade_level,enrollment
0,100001,Region I,Ilocos Norte,Public,SY 2022-2023,Kindergarten,6
1,100002,Region I,Ilocos Norte,Public,SY 2022-2023,Kindergarten,56
2,100003,Region I,Ilocos Norte,Public,SY 2022-2023,Kindergarten,23


CPU times: user 11.3 s, sys: 1.1 s, total: 12.4 s
Wall time: 5.69 s


## 1.2. ESC beneficiaries

In [4]:
%%time
# Load parquet
path_parquet = os.path.join(cf.get_path('processed_data'), 'esc_beneficiaries.parquet')
df_esc = pd.read_parquet(path_parquet)

print(df_esc.shape)
display(df_esc.head(3))

(2681668, 13)


,deped_school_id,lrn,esc_school_id,school_name,grade,billing_statement_number,esc_subsidy_amount,sheet_name,school_year,grade_level,lrn_validated,lrn_school_id,region
0,475511,133526130177,1603520,"Adiong Memorial College Foundation, Inc.",9,ESC-227153,9000.0,BARMM,SY 2022-2023,Grade 9,133526130177,133526,BARMM
1,475511,133536150012,1603520,"Adiong Memorial College Foundation, Inc.",7,ESC-227153,9000.0,BARMM,SY 2022-2023,Grade 7,133536150012,133536,BARMM
2,475511,133536150011,1603520,"Adiong Memorial College Foundation, Inc.",7,ESC-227153,9000.0,BARMM,SY 2022-2023,Grade 7,133536150011,133536,BARMM


CPU times: user 23.7 s, sys: 1.86 s, total: 25.6 s
Wall time: 12.6 s


In [5]:
def reorder_grades(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reorders a MultiIndex DataFrame's columns based on grade levels.
    Kindergarten will be placed before all other grades.

    Args:
        df: A pandas DataFrame with a MultiIndex column where the first
            level is 'school_year' and the second level is 'grade_level'.

    Returns:
        A new DataFrame with the grade levels in the correct order.
    """
    # Check if the DataFrame has the expected MultiIndex
    if not isinstance(df.columns, pd.MultiIndex) or 'school_year' not in df.columns.names or 'grade_level' not in df.columns.names:
        raise ValueError("DataFrame must have a MultiIndex with 'school_year' and 'grade_level' as column names.")

    # Extract unique school years and grade levels
    school_years = df.columns.get_level_values('school_year').unique().tolist()
    grade_levels = df.columns.get_level_values('grade_level').unique().tolist()

    # Sort the grade levels using a custom key to handle "Kindergarten"
    def sort_key(grade_str):
        if 'Kindergarten' in grade_str:
            return 0
        return int(grade_str.split()[1])

    grade_levels_sorted = sorted(
        grade_levels,
        key=sort_key
    )

    # Create the new MultiIndex order programmatically
    new_order = [(sy, gl) for sy in school_years for gl in grade_levels_sorted]

    # Reindex the DataFrame with the new order
    df_reordered = df.reindex(columns=new_order)

    return df_reordered

In [6]:
stime = time.time()

df_bfs = df_esc.copy()

df_bfs = df_bfs.rename(
    columns={
        'deped_school_id':'school_id',
    }
)
df_bfs['school_id'] = df_bfs['school_id'].astype('string')

pvt_bfs_schs = df_bfs.pivot_table(
    index=['school_id'],
    columns=['school_year','grade_level'],
    values='lrn',
    aggfunc='count',
    observed=False
)

pvt_reordered = reorder_grades(pvt_bfs_schs)

etime = np.round(time.time() - stime, decimals=2)
print(f"Time elapsed: {etime} seconds")

# To view the result:
display(pvt_reordered.head())

Time elapsed: 8.86 seconds


school_year SY 2022-2023                          SY 2023-2024          \
grade_level      Grade 7 Grade 8 Grade 9 Grade 10      Grade 7 Grade 8   
school_id                                                                
0                      9      10      10       16            0       0   
302762                 7       4       5        5            0       0   
400001                10       8       7        9            9      10   
400002                41      49      48       52           21      33   
400003                65      60      69       66           27      60   

school_year                  SY 2024-2025                           
grade_level Grade 9 Grade 10      Grade 7 Grade 8 Grade 9 Grade 10  
school_id                                                           
0                 0        0            0       0       0        0  
302762            0        0            0       0       0        0  
400001           10        7           12       8       9        9  
400002           43       41           21      19      31       34  
400003           52       65           51      26      43       50

## 1.3. Seats

In [7]:
!ls data/processed

esc_beneficiaries.parquet
kb_class.pkl
psgc_shapefiles.pkl
raw_validation_sheets
SY_2022_2023_School_Level_Data_on_Official_Enrollment.csv
SY_2023_2024_School_Level_Data_on_Official_Enrollment.csv
SY_2024_2025_School_Level_Data_on_Official_Enrollment.csv


In [8]:
loadpath = os.path.join(cf.get_path('processed_data'), 'kb_class.pkl')
with open(loadpath, mode='rb') as file:
    kb_class = pickle.load(file)

In [9]:
seats_private = kb_class.private_seats.copy()
seats_private['school id'] = seats_private['school id'].astype('string')
print(seats_private.shape)
display(seats_private.head(3))

(32042, 4)


,school id,furniture_category,furniture_count,level_of_education
0,400003,sets of chairs and tables kinder,14,Kindergarten
1,400005,sets of chairs and tables kinder,23,Kindergarten
2,400011,sets of chairs and tables kinder,40,Kindergarten


In [10]:
seats_public = kb_class.public_seats.copy()
seats_public['school id'] = seats_public['school id'].astype('string')
print(seats_public.shape)
display(seats_public.head(3))

(54356, 3)


,school id,level_of_education,count_seats
0,100001,Elementary,195
1,100002,Elementary,731
2,100003,Elementary,192


In [15]:
pvt_seats = seats_private.pivot_table(
    index='school id',
    columns='level_of_education',
    values='furniture_count',
    aggfunc='sum'
)

es_seats = pvt_seats[['Elementary']].copy()
print(es_seats.shape)
display(es_seats.head(3))

(10811, 1)


level_of_education,Elementary
school id,
104304,NaN
104357,NaN
137153,21


# 2. Discovery

## 2.1. Pivot Enrollment

In [12]:
display(df_enr.head(1))

,school_id,old_region,division,sector,school_year,grade_level,enrollment
0,100001,Region I,Ilocos Norte,Public,SY 2022-2023,Kindergarten,6


In [13]:
%%time
pvt_enr = df_enr.pivot_table(
    index=['school_id'],
    columns=['school_year','grade_level'],
    values='enrollment',
    aggfunc='sum',
    observed=False
)

pvt_reord = reorder_grades(pvt_enr)

# Extract unique school years and grade levels
school_years = pvt_reord.columns.get_level_values('school_year').unique().tolist()
grade_levels = pvt_reord.columns.get_level_values('grade_level').unique().tolist()

es_lvls = ['Grade 1','Grade 2','Grade 3','Grade 4','Grade 5','Grade 6']
drop_cols = []
for sy in school_years:
    for lvl in grade_levels:
        col = (sy, lvl)

        if lvl not in es_lvls:
            drop_cols.append(col)

pvt_trim = pvt_reord.drop(columns=drop_cols)

print(pvt_trim.shape)
display(pvt_trim.head())

(61408, 18)


school_year SY 2022-2023                                         SY 2023-2024  \
grade_level      Grade 1 Grade 2 Grade 3 Grade 4 Grade 5 Grade 6      Grade 1   
school_id                                                                       
100000              32.0    42.0    35.0    37.0    38.0    40.0         54.0   
100001              13.0     6.0     3.0    10.0    10.0     6.0          5.0   
100002              46.0    58.0    51.0    70.0    68.0    47.0         58.0   
100003              16.0    19.0    16.0    27.0    19.0    16.0         21.0   
100004              10.0    14.0    12.0    11.0    13.0    12.0          9.0   

school_year                                         SY 2024-2025          \
grade_level Grade 2 Grade 3 Grade 4 Grade 5 Grade 6      Grade 1 Grade 2   
school_id                                                                  
100000         31.0    41.0    36.0    38.0    39.0         40.0    54.0   
100001         12.0     6.0     4.0    11.0    11.0          7.0     6.0   
100002         47.0    55.0    53.0    69.0    69.0         55.0    60.0   
100003         15.0    17.0    15.0    26.0    16.0         13.0    23.0   
100004         11.0    14.0    12.0    14.0    14.0         17.0     9.0   

school_year                                  
grade_level Grade 3 Grade 4 Grade 5 Grade 6  
school_id                                    
100000         33.0    39.0    36.0    38.0  
100001         12.0     7.0     4.0    12.0  
100002         46.0    51.0    52.0    68.0  
100003         13.0    17.0    13.0    24.0  
100004         10.0    13.0    11.0    13.0

CPU times: user 12.8 s, sys: 1.85 s, total: 14.7 s
Wall time: 14.8 s


## 2.2. Enrollment x ESC

In [34]:
display(pvt_reordered.head())

school_year SY 2022-2023                          SY 2023-2024          \
grade_level      Grade 7 Grade 8 Grade 9 Grade 10      Grade 7 Grade 8   
school_id                                                                
0                      9      10      10       16            0       0   
302762                 7       4       5        5            0       0   
400001                10       8       7        9            9      10   
400002                41      49      48       52           21      33   
400003                65      60      69       66           27      60   

school_year                  SY 2024-2025                           
grade_level Grade 9 Grade 10      Grade 7 Grade 8 Grade 9 Grade 10  
school_id                                                           
0                 0        0            0       0       0        0  
302762            0        0            0       0       0        0  
400001           10        7           12       8       9        9  
400002           43       41           21      19      31       34  
400003           52       65           51      26      43       50

In [75]:
# Only get the IDs of schools with beneficiaries for SY 24-25
adf = pvt_reordered.iloc[:, -4:] # SY 24-25 columns
bdf = adf[adf.gt(0).any(axis=1)]
bfs_idxs = bdf.index.tolist()
print(len(bfs_idxs))

3533


In [76]:
# bfs_idxs = pvt_reordered.index[1:].tolist()
enr_idxs = pvt_trim.index.tolist()
# print(len(bfs_idxs))
print(len(enr_idxs))

61408


In [77]:
diff_idxs = list(set(bfs_idxs).difference(set(enr_idxs)))
print(diff_idxs)
print(len(diff_idxs))

[]
0


In [23]:
mask = pvt_trim.index.isin(bfs_idxs)

print("="*50+"ENROLLMENT"+"="*50)
display(pvt_trim.loc[mask].head())

==================================================ENROLLMENT==================================================


school_year SY 2022-2023                          SY 2023-2024          \
grade_level      Grade 7 Grade 8 Grade 9 Grade 10      Grade 7 Grade 8   
school_id                                                                
302762             247.0   239.0   267.0    259.0        242.0   231.0   
400001               9.0    10.0     7.0     11.0          9.0    10.0   
400002              44.0    52.0    51.0     72.0         21.0    33.0   
400003              66.0    62.0    68.0     68.0         27.0    60.0   
400004              93.0    91.0   105.0     83.0         96.0    90.0   

school_year                  SY 2024-2025                           
grade_level Grade 9 Grade 10      Grade 7 Grade 8 Grade 9 Grade 10  
school_id                                                           
302762        239.0    241.0        265.0   214.0   228.0    223.0  
400001         10.0      8.0         12.0     9.0    12.0     10.0  
400002         47.0     45.0         32.0    21.0    32.0     46.0  
400003         52.0     63.0         51.0    27.0    45.0     50.0  
400004         88.0     97.0         82.0    95.0    93.0     79.0

In [24]:
print("="*50+"ESC BENEFICIARIES"+"="*50)
display(pvt_reordered[pvt_reordered.index.isin(bfs_idxs)].head())

==================================================ESC BENEFICIARIES==================================================


school_year SY 2022-2023                          SY 2023-2024          \
grade_level      Grade 7 Grade 8 Grade 9 Grade 10      Grade 7 Grade 8   
school_id                                                                
302762                 7       4       5        5            0       0   
400001                10       8       7        9            9      10   
400002                41      49      48       52           21      33   
400003                65      60      69       66           27      60   
400004                93      91     105       83           96      87   

school_year                  SY 2024-2025                           
grade_level Grade 9 Grade 10      Grade 7 Grade 8 Grade 9 Grade 10  
school_id                                                           
302762            0        0            0       0       0        0  
400001           10        7           12       8       9        9  
400002           43       41           21      19      31       34  
400003           52       65           51      26      43       50  
400004           84       95           81      86      88       74

### Grade 7 only

In [25]:
def get_certain_grade_level_columns(df_pvt, columns):
    # Extract unique school years and grade levels
    school_years = df_pvt.columns.get_level_values('school_year').unique().tolist()
    grade_levels = df_pvt.columns.get_level_values('grade_level').unique().tolist()
    
    drop_cols = []
    for sy in school_years:
        for lvl in grade_levels:
            col = (sy, lvl)
    
            if lvl not in columns:
                drop_cols.append(col)

    return df_pvt.copy().drop(columns=drop_cols)

In [26]:
tmp_enr = pvt_trim[pvt_trim.index.isin(bfs_idxs)]
enr_g7 = get_certain_grade_level_columns(
    tmp_enr,
    columns=['Grade 7']
)

tmp_esc = pvt_reordered[pvt_reordered.index.isin(bfs_idxs)]
esc_g7 = get_certain_grade_level_columns(
    tmp_esc,
    columns=['Grade 7']
)

In [27]:
print("="*20+"ENROLLMENT"+"="*20)
display(enr_g7.head(5))
print()
print("="*20+"BENEFICIARIES"+"="*20)
display(esc_g7.head(5))

====================ENROLLMENT====================


school_year,SY 2022-2023,SY 2023-2024,SY 2024-2025
grade_level,Grade 7,Grade 7,Grade 7
school_id,,,
302762,247.0,242.0,265.0
400001,9.0,9.0,12.0
400002,44.0,21.0,32.0
400003,66.0,27.0,51.0
400004,93.0,96.0,82.0



====================BENEFICIARIES====================


school_year,SY 2022-2023,SY 2023-2024,SY 2024-2025
grade_level,Grade 7,Grade 7,Grade 7
school_id,,,
302762,7,0,0
400001,10,9,12
400002,41,21,21
400003,65,27,51
400004,93,96,81


### With Seats

In [16]:
display(es_seats.head(1))

level_of_education,Elementary
school id,
104304,NaN


In [17]:
seats = es_seats.copy()

# Correctly create a two-layer MultiIndex
seats.columns = pd.MultiIndex.from_tuples([('SY 2023-2024', 'Elementary')])

enr_seats = enr_g7.join(seats)
esc_seats = esc_g7.join(seats)
print(enr_seats.shape)

NameError: name 'enr_g7' is not defined

In [30]:
print("="*20+"ENROLLMENT"+"="*20)
display(enr_seats.head(5))
print()
print("="*20+"BENEFICIARIES"+"="*20)
display(esc_seats.head(5))

====================ENROLLMENT====================


,SY 2022-2023,SY 2023-2024,SY 2024-2025,SY 2023-2024
,Grade 7,Grade 7,Grade 7,Junior High School
school_id,,,,
302762,247.0,242.0,265.0,NaN
400001,9.0,9.0,12.0,36
400002,44.0,21.0,32.0,250
400003,66.0,27.0,51.0,177
400004,93.0,96.0,82.0,369



====================BENEFICIARIES====================


,SY 2022-2023,SY 2023-2024,SY 2024-2025,SY 2023-2024
,Grade 7,Grade 7,Grade 7,Junior High School
school_id,,,,
302762,7,0,0,NaN
400001,10,9,12,36
400002,41,21,21,250
400003,65,27,51,177
400004,93,96,81,369


In [31]:
enr_seats.isna().sum()

SY 2022-2023  Grade 7                 3
SY 2023-2024  Grade 7                25
SY 2024-2025  Grade 7                55
SY 2023-2024  Junior High School    301
dtype: int64

# 3. Shenanigans

## 3.1. Enr + Seats

In [18]:
print(pvt_reord.shape)
display(pvt_reord.head(1))

(61408, 39)


school_year SY 2022-2023                                                  \
grade_level Kindergarten Grade 1 Grade 2 Grade 3 Grade 4 Grade 5 Grade 6   
school_id                                                                  
100000              58.0    32.0    42.0    35.0    37.0    38.0    40.0   

school_year                          ... SY 2024-2025                          \
grade_level Grade 7 Grade 8 Grade 9  ...      Grade 3 Grade 4 Grade 5 Grade 6   
school_id                            ...                                        
100000          0.0     0.0     0.0  ...         33.0    39.0    36.0    38.0   

school_year                                                     
grade_level Grade 7 Grade 8 Grade 9 Grade 10 Grade 11 Grade 12  
school_id                                                       
100000          0.0     0.0     0.0      0.0      0.0      0.0  

[1 rows x 39 columns]

In [19]:
enr_pvt = pvt_reord.copy()

school_years = enr_pvt.columns.get_level_values('school_year').unique().tolist()
grade_levels = enr_pvt.columns.get_level_values('grade_level').unique().tolist()

flat_cols = []
for yr in school_years:
    for lvl in grade_levels:
        n_yr = '_'.join(yr.lower().split(' '))
        n_lvl = '_'.join(lvl.lower().split(' '))
        n_col = n_yr + '_' + n_lvl
        flat_cols.append(n_col)

enr_pvt.columns = flat_cols
print(enr_pvt.shape)
display(enr_pvt.head(2))

(61408, 39)


,sy_2022-2023_kindergarten,sy_2022-2023_grade_1,sy_2022-2023_grade_2,sy_2022-2023_grade_3,sy_2022-2023_grade_4,sy_2022-2023_grade_5,sy_2022-2023_grade_6,sy_2022-2023_grade_7,sy_2022-2023_grade_8,sy_2022-2023_grade_9,...,sy_2024-2025_grade_3,sy_2024-2025_grade_4,sy_2024-2025_grade_5,sy_2024-2025_grade_6,sy_2024-2025_grade_7,sy_2024-2025_grade_8,sy_2024-2025_grade_9,sy_2024-2025_grade_10,sy_2024-2025_grade_11,sy_2024-2025_grade_12
school_id,,,,,,,,,,,,,,,,,,,,,
100000,58.0,32.0,42.0,35.0,37.0,38.0,40.0,0.0,0.0,0.0,...,33.0,39.0,36.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0
100001,6.0,13.0,6.0,3.0,10.0,10.0,6.0,0.0,0.0,0.0,...,12.0,7.0,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
pvt_pub_seats = seats_public.pivot_table(
    index='school id',
    columns='level_of_education',
    values='count_seats',
    aggfunc='sum'
)
pvt_pub_seats = pvt_pub_seats.reset_index()
pvt_pub_seats.columns.name = ''

display(pvt_pub_seats.head(1))

,school id,Elementary,Junior High School,Senior High School
0,100000,189,NaN,NaN


In [21]:
pvt_priv_seats = seats_private.pivot_table(
    index='school id',
    columns='level_of_education',
    values='furniture_count',
    aggfunc='sum'
)
pvt_priv_seats = pvt_priv_seats.drop(columns=['Kindergarten'])

pvt_priv_seats = pvt_priv_seats.reset_index()
pvt_priv_seats.columns.name = ''

display(pvt_priv_seats.head(1))

,school id,Elementary,Junior High School,Senior High School
0,104304,NaN,NaN,NaN


In [22]:
all_seats = pd.concat([pvt_pub_seats, pvt_priv_seats])
all_seats = all_seats.rename(columns={'school id':'school_id'})
all_seats['school_id'] = all_seats['school_id'].astype('string')
all_seats = all_seats.set_index('school_id')

all_seats.columns = ['seats_'+'_'.join(col.lower().split(' ')) for col in all_seats.columns]

print(all_seats.shape)
display(all_seats.head())

(56581, 3)


,seats_elementary,seats_junior_high_school,seats_senior_high_school
school_id,,,
100000,189,NaN,NaN
100001,195,NaN,NaN
100002,731,NaN,NaN
100003,192,NaN,NaN
100004,134,NaN,NaN


In [23]:
enr_info = kb_class.enrollment_info.copy()
rel_cols = ['region','division','school_id','province','municipality','barangay','sector','modified coc']
enr_info = enr_info[rel_cols].set_index('school_id')

jn_enr_sts = enr_pvt.join([enr_info, all_seats])
print(jn_enr_sts.shape)
display(jn_enr_sts.head())

(61408, 49)


,sy_2022-2023_kindergarten,sy_2022-2023_grade_1,sy_2022-2023_grade_2,sy_2022-2023_grade_3,sy_2022-2023_grade_4,sy_2022-2023_grade_5,sy_2022-2023_grade_6,sy_2022-2023_grade_7,sy_2022-2023_grade_8,sy_2022-2023_grade_9,...,region,division,province,municipality,barangay,sector,modified coc,seats_elementary,seats_junior_high_school,seats_senior_high_school
school_id,,,,,,,,,,,,,,,,,,,,,
100000,58.0,32.0,42.0,35.0,37.0,38.0,40.0,0.0,0.0,0.0,...,Region I,San Carlos City,PANGASINAN,SAN CARLOS CITY,BALAYONG,Public,Purely ES,189,NaN,NaN
100001,6.0,13.0,6.0,3.0,10.0,10.0,6.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,LIBTONG,Public,Purely ES,195,NaN,NaN
100002,56.0,46.0,58.0,51.0,70.0,68.0,47.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,SANTA RITA (POB.),Public,Purely ES,731,NaN,NaN
100003,23.0,16.0,19.0,16.0,27.0,19.0,16.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,BUYON,Public,Purely ES,192,NaN,NaN
100004,10.0,10.0,14.0,12.0,11.0,13.0,12.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,GANAGAN,Public,Purely ES,134,NaN,NaN


### Only ES mCoC

In [24]:
display(jn_enr_sts['modified coc'].unique())

array(['Purely ES', nan, 'Purely JHS', 'JHS with SHS', 'Purely SHS',
       'All Offering', 'ES and JHS'], dtype=object)

In [27]:
es_coc = ['Purely ES','All Offering','ES and JHS']
target_sectors = ['Public','Private']
mask = (
    (jn_enr_sts['modified coc'].isin(es_coc))
    & (jn_enr_sts['sector'].isin(target_sectors))
)
df_es = jn_enr_sts.loc[mask].copy()

print(df_es.shape)

(49270, 49)


In [28]:
df_es.isna().sum()

sy_2022-2023_kindergarten      515
sy_2022-2023_grade_1           515
sy_2022-2023_grade_2           515
sy_2022-2023_grade_3           515
sy_2022-2023_grade_4           515
sy_2022-2023_grade_5           515
sy_2022-2023_grade_6           515
sy_2022-2023_grade_7           515
sy_2022-2023_grade_8           515
sy_2022-2023_grade_9           515
sy_2022-2023_grade_10          515
sy_2022-2023_grade_11          515
sy_2022-2023_grade_12          515
sy_2023-2024_kindergarten        0
sy_2023-2024_grade_1             0
sy_2023-2024_grade_2             0
sy_2023-2024_grade_3             0
sy_2023-2024_grade_4             0
sy_2023-2024_grade_5             0
sy_2023-2024_grade_6             0
sy_2023-2024_grade_7             0
sy_2023-2024_grade_8             0
sy_2023-2024_grade_9             0
sy_2023-2024_grade_10            0
sy_2023-2024_grade_11            0
sy_2023-2024_grade_12            0
sy_2024-2025_kindergarten      514
sy_2024-2025_grade_1           514
sy_2024-2025_grade_2

In [29]:
df_es.head()

,sy_2022-2023_kindergarten,sy_2022-2023_grade_1,sy_2022-2023_grade_2,sy_2022-2023_grade_3,sy_2022-2023_grade_4,sy_2022-2023_grade_5,sy_2022-2023_grade_6,sy_2022-2023_grade_7,sy_2022-2023_grade_8,sy_2022-2023_grade_9,...,region,division,province,municipality,barangay,sector,modified coc,seats_elementary,seats_junior_high_school,seats_senior_high_school
school_id,,,,,,,,,,,,,,,,,,,,,
100000,58.0,32.0,42.0,35.0,37.0,38.0,40.0,0.0,0.0,0.0,...,Region I,San Carlos City,PANGASINAN,SAN CARLOS CITY,BALAYONG,Public,Purely ES,189,NaN,NaN
100001,6.0,13.0,6.0,3.0,10.0,10.0,6.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,LIBTONG,Public,Purely ES,195,NaN,NaN
100002,56.0,46.0,58.0,51.0,70.0,68.0,47.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,SANTA RITA (POB.),Public,Purely ES,731,NaN,NaN
100003,23.0,16.0,19.0,16.0,27.0,19.0,16.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,BUYON,Public,Purely ES,192,NaN,NaN
100004,10.0,10.0,14.0,12.0,11.0,13.0,12.0,0.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,GANAGAN,Public,Purely ES,134,NaN,NaN


In [30]:
df_es.to_csv('output/seats_imputation_data_es.csv', index=True)

## 3.2. Load imputed

In [32]:
df_imp = pd.read_csv('output/seats_imputed_data_es.csv')
print(df_imp.shape)

(49270, 50)


In [33]:
df_imp.head()

,school_id,sy_2022-2023_kindergarten,sy_2022-2023_grade_1,sy_2022-2023_grade_2,sy_2022-2023_grade_3,sy_2022-2023_grade_4,sy_2022-2023_grade_5,sy_2022-2023_grade_6,sy_2022-2023_grade_7,sy_2022-2023_grade_8,...,region,division,province,municipality,barangay,sector,modified coc,seats_elementary,seats_junior_high_school,seats_senior_high_school
0,100000,58.0,32.0,42.0,35.0,37.0,38.0,40.0,0.0,0.0,...,Region I,San Carlos City,PANGASINAN,SAN CARLOS CITY,BALAYONG,Public,Purely ES,189.0,NaN,NaN
1,100001,6.0,13.0,6.0,3.0,10.0,10.0,6.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,LIBTONG,Public,Purely ES,195.0,NaN,NaN
2,100002,56.0,46.0,58.0,51.0,70.0,68.0,47.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,SANTA RITA (POB.),Public,Purely ES,731.0,NaN,NaN
3,100003,23.0,16.0,19.0,16.0,27.0,19.0,16.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,BUYON,Public,Purely ES,192.0,NaN,NaN
4,100004,10.0,10.0,14.0,12.0,11.0,13.0,12.0,0.0,0.0,...,Region I,Ilocos Norte,ILOCOS NORTE,BACARRA,GANAGAN,Public,Purely ES,134.0,NaN,NaN


In [35]:
df_imp.columns

Index(['school_id', 'sy_2022-2023_kindergarten', 'sy_2022-2023_grade_1',
       'sy_2022-2023_grade_2', 'sy_2022-2023_grade_3', 'sy_2022-2023_grade_4',
       'sy_2022-2023_grade_5', 'sy_2022-2023_grade_6', 'sy_2022-2023_grade_7',
       'sy_2022-2023_grade_8', 'sy_2022-2023_grade_9', 'sy_2022-2023_grade_10',
       'sy_2022-2023_grade_11', 'sy_2022-2023_grade_12',
       'sy_2023-2024_kindergarten', 'sy_2023-2024_grade_1',
       'sy_2023-2024_grade_2', 'sy_2023-2024_grade_3', 'sy_2023-2024_grade_4',
       'sy_2023-2024_grade_5', 'sy_2023-2024_grade_6', 'sy_2023-2024_grade_7',
       'sy_2023-2024_grade_8', 'sy_2023-2024_grade_9', 'sy_2023-2024_grade_10',
       'sy_2023-2024_grade_11', 'sy_2023-2024_grade_12',
       'sy_2024-2025_kindergarten', 'sy_2024-2025_grade_1',
       'sy_2024-2025_grade_2', 'sy_2024-2025_grade_3', 'sy_2024-2025_grade_4',
       'sy_2024-2025_grade_5', 'sy_2024-2025_grade_6', 'sy_2024-2025_grade_7',
       'sy_2024-2025_grade_8', 'sy_2024-2025_grade_9', 'sy_

In [36]:
rel_cols = (
    ['school_id'] + 
    df_imp.loc[:, 'sy_2024-2025_grade_1':'sy_2024-2025_grade_6'].columns.tolist() +
    df_imp.loc[:, 'region':'seats_elementary'].columns.tolist()
)
# rel_cols.remove('seats_senior_high_school')

df_rel = df_imp[rel_cols].copy()

# PRIVATE sector only
mask = df_rel['sector'] == 'Private'
df_rel = df_rel.loc[mask].copy()

# Get aggregate ES enrollment
df_rel['total_enrollment_es'] = df_rel.loc[:,'sy_2024-2025_grade_1':'sy_2024-2025_grade_6'].sum(axis=1)

# Free seats
df_rel['free_seats_es'] = df_rel['seats_elementary'] - df_rel['total_enrollment_es']
df_rel['free_seats_es'] = df_rel['free_seats_es'].apply(lambda x: 0 if x < 0 else x)

df_rel['school_id'] = df_rel['school_id'].astype('string')

# Tag PRIVATE schools with ESC beneficiaries

print(df_rel.shape)
display(df_rel.head())

(9924, 17)


,school_id,sy_2024-2025_grade_1,sy_2024-2025_grade_2,sy_2024-2025_grade_3,sy_2024-2025_grade_4,sy_2024-2025_grade_5,sy_2024-2025_grade_6,region,division,province,municipality,barangay,sector,modified coc,seats_elementary,total_enrollment_es,free_seats_es
36999,400000,0.0,0.0,0.0,0.0,0.0,0.0,CARAGA,Dinagat Island,DINAGAT ISLANDS,SAN JOSE (Capital),AURELIO,Private,Purely ES,49.0,0.0,49.0
37000,400003,18.0,11.0,8.0,11.0,6.0,19.0,Region I,Ilocos Norte,ILOCOS NORTE,BADOC,GARRETA (POB.),Private,All Offering,63.0,73.0,0.0
37001,400005,26.0,8.0,13.0,8.0,8.0,13.0,Region I,Ilocos Norte,ILOCOS NORTE,BADOC,GARRETA (POB.),Private,Purely ES,70.0,76.0,0.0
37002,400008,33.0,23.0,24.0,22.0,12.0,14.0,Region I,Batac City,ILOCOS NORTE,CITY OF BATAC,VALDEZ POB. (CAOAYAN),Private,All Offering,110.0,128.0,0.0
37003,400010,29.0,20.0,21.0,30.0,18.0,18.0,Region I,Batac City,ILOCOS NORTE,CITY OF BATAC,VALDEZ POB. (CAOAYAN),Private,All Offering,153.0,136.0,17.0


In [62]:
# # Sanity check - check count of Private whose modified COC has JHS (~5,620)
# jhs_coc = ['Purely JHS','JHS with SHS','All Offering','ES and JHS']
# mask = (
#     (enr_info['sector'] == 'Private')
#     & (enr_info['modified coc'].isin(jhs_coc))
# )
# enr_info.loc[mask]

In [42]:
fdf_es.head(1)

,school_id,sy_2024-2025_grade_1,sy_2024-2025_grade_2,sy_2024-2025_grade_3,sy_2024-2025_grade_4,sy_2024-2025_grade_5,sy_2024-2025_grade_6,region,division,province,municipality,barangay,sector,modified coc,seats_elementary,total_enrollment_es,free_seats_es


In [51]:
fdf_es = df_rel.copy()
print(fdf_es.shape)

# Include schools that have "valid" enrollment values
adf = fdf_es.set_index('school_id')
adf = adf.loc[:, 'sy_2024-2025_grade_1':'sy_2024-2025_grade_6'] # SY 24-25 columns
bdf = adf[adf.gt(0).any(axis=1)]
bdf_idxs = bdf.index.tolist()

fdf_es = fdf_es[fdf_es['school_id'].isin(bdf_idxs)].copy()

# tag JHS seats that were imputed
imp_idxs = df_es[df_es['seats_elementary'].isna()].index.tolist()
mask = fdf_es['school_id'].isin(imp_idxs)
fdf_es.loc[mask, 'es_seats_was_imputed'] = 1

display(fdf_es)

(9924, 17)


,school_id,sy_2024-2025_grade_1,sy_2024-2025_grade_2,sy_2024-2025_grade_3,sy_2024-2025_grade_4,sy_2024-2025_grade_5,sy_2024-2025_grade_6,region,division,province,municipality,barangay,sector,modified coc,seats_elementary,total_enrollment_es,free_seats_es,es_seats_was_imputed
37000,400003,18.0,11.0,8.0,11.0,6.0,19.0,Region I,Ilocos Norte,ILOCOS NORTE,BADOC,GARRETA (POB.),Private,All Offering,63.0,73.0,0.0,NaN
37001,400005,26.0,8.0,13.0,8.0,8.0,13.0,Region I,Ilocos Norte,ILOCOS NORTE,BADOC,GARRETA (POB.),Private,Purely ES,70.0,76.0,0.0,NaN
37002,400008,33.0,23.0,24.0,22.0,12.0,14.0,Region I,Batac City,ILOCOS NORTE,CITY OF BATAC,VALDEZ POB. (CAOAYAN),Private,All Offering,110.0,128.0,0.0,NaN
37003,400010,29.0,20.0,21.0,30.0,18.0,18.0,Region I,Batac City,ILOCOS NORTE,CITY OF BATAC,VALDEZ POB. (CAOAYAN),Private,All Offering,153.0,136.0,17.0,NaN
37004,400011,26.0,25.0,21.0,22.0,9.0,24.0,Region I,Ilocos Norte,ILOCOS NORTE,DINGRAS,MADAMBA (POB.),Private,ES and JHS,137.0,127.0,10.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46916,499002,1.0,2.0,0.0,2.0,0.0,0.0,Region X,Tangub City,MISAMIS OCCIDENTAL,TANGUB CITY,GARANG,Private,Purely ES,6.0,5.0,1.0,NaN
46917,499003,9.0,8.0,6.0,4.0,5.0,4.0,Region X,Tangub City,MISAMIS OCCIDENTAL,TANGUB CITY,MALORO,Private,Purely ES,28.0,36.0,0.0,NaN
46918,499004,5.0,6.0,2.0,5.0,7.0,1.0,Region X,Tangub City,MISAMIS OCCIDENTAL,TANGUB CITY,SANTA CRUZ,Private,ES and JHS,11.0,26.0,0.0,NaN
46919,499005,4.0,7.0,7.0,3.0,2.0,6.0,Region X,Tangub City,MISAMIS OCCIDENTAL,TANGUB CITY,BARANGAY III- MARKET KALUBIAN (POB.),Private,ES and JHS,59.0,29.0,30.0,1.0


In [52]:
fdf_es.to_csv('output/absorptive_capacity_matrix_es.csv', index=False)